# Produce Category Strategy Card

This analysis evaluates customer purchasing patterns in Instacart's Produce department, comparing Organic vs. Conventional produce segments, conducting a berry promotion campaign analysis, and identifying top-selling organic produce anchors to render an executive strategy card.

## Load the datasets

Load the Instacart product catalog and order prior purchase history.

In [6]:
import os
import kagglehub
import pandas as pd
from IPython.display import HTML, display

# Load Instacart Market Basket dataset
DATA_PATH = kagglehub.dataset_download("psparks/instacart-market-basket-analysis")

products = pd.read_csv(os.path.join(DATA_PATH, "products.csv"))
order_prior = pd.read_csv(os.path.join(DATA_PATH, "order_products__prior.csv"))

print("Products loaded:", f"{len(products):,}")
print("Prior order items loaded:", f"{len(order_prior):,}")


Products loaded: 49,688
Prior order items loaded: 32,434,489


## Compare Organic vs. Conventional Produce segments

Classify Produce department items into Organic vs. Conventional segments and compare product counts, total order volumes, and mean reorder rates.

In [7]:
# Macro Segment Comparison (Organic vs. Conventional)
produce_prods = products[products["department_id"] == 4].copy()
produce_prods["is_organic"] = produce_prods["product_name"].str.contains("Organic", case=False)

merged_produce = order_prior.merge(produce_prods, on="product_id")

summary_df = merged_produce.groupby("is_organic").agg(
    total_orders=("order_id", "count"),
    reorder_rate=("reordered", "mean"),
    product_count=("product_id", "nunique")
).reset_index()

print("Organic vs. Conventional Produce Segment Comparison:")
print(summary_df.to_string(index=False))

# Extract metrics for downstream HTML card rendering
org_row = summary_df[summary_df["is_organic"] == True].iloc[0]
conv_row = summary_df[summary_df["is_organic"] == False].iloc[0]

org_prods = int(org_row["product_count"])
org_orders = int(org_row["total_orders"])
org_reorder = float(org_row["reorder_rate"])

conv_prods = int(conv_row["product_count"])
conv_orders = int(conv_row["total_orders"])
conv_reorder = float(conv_row["reorder_rate"])


Organic vs. Conventional Produce Segment Comparison:
 is_organic  total_orders  reorder_rate  product_count
      False       4366227      0.625183           1211
       True       5113064      0.671030            473


## Conduct Berry Promotion Campaign Comparison

Compare order volume and reorder rates for Organic Strawberries (21137) vs. Ordinary Strawberries (16797) for the upcoming marketing promotion.

In [8]:
# Berry Campaign Comparison
org_straw = order_prior[order_prior["product_id"] == 21137]
conv_straw = order_prior[order_prior["product_id"] == 16797]

print("Berry Promotion Campaign Comparison:")
print("Organic Strawberries (21137):", f"{len(org_straw):,} orders,", f"{org_straw['reordered'].mean():.2%} reorder rate")
print("Strawberries Ordinary (16797):", f"{len(conv_straw):,} orders,", f"{conv_straw['reordered'].mean():.2%} reorder rate")

# Extract metrics for downstream HTML card rendering
org_straw_orders = len(org_straw)
org_straw_reorder = float(org_straw["reordered"].mean())

conv_straw_orders = len(conv_straw)
conv_straw_reorder = float(conv_straw["reordered"].mean())


Berry Promotion Campaign Comparison:
Organic Strawberries (21137): 264,683 orders, 77.77% reorder rate
Strawberries Ordinary (16797): 142,951 orders, 69.82% reorder rate


## Select Category Staples and Fetch Product IDs

Define a helper function `fetch_product_id(name)`, identify conventional banana baseline (24852), and assign the signature organic produce staple.

In [9]:
def fetch_product_id(product_name):
    exact = products[products["product_name"].str.strip().str.casefold() == product_name.strip().casefold()]
    if not exact.empty:
        return int(exact.iloc[0]["product_id"])
    contains = products[products["product_name"].str.contains(product_name, case=False, na=False)]
    return int(contains.iloc[0]["product_id"]) if not contains.empty else None

banana_id = fetch_product_id("Banana")

# Organic Produce Anchor Assignment (Side Effect - Parametric Assignment + Auto-Match)
organic_title = "Organic Banana"
organic_id = fetch_product_id(organic_title)

print(f"Conventional Banana Baseline Staple: Banana (Product ID: {banana_id})")
print(f"Selected Organic Anchor Staple: {organic_title} (Matched Product ID: {organic_id})")

nb2_outlook = "Organic produce generates higher total order volume (5.11M vs. 4.37M) and superior reorder rates (67.10% vs. 62.52%) compared to conventional produce. Organic Strawberries demonstrate strong customer loyalty with a 77.77% reorder rate. Prioritizing marketing and placement for organic produce, anchored by Organic Banana as a category staple, will maximize customer engagement."


Conventional Banana Baseline Staple: Banana (Product ID: 24852)
Selected Organic Anchor Staple: Organic Banana (Matched Product ID: 37067)


## Render the strategy card

Render the parallel staple cards, strawberry campaign comparison box, and strategic recommendation as an HTML strategy card.

In [10]:
card_html = f"""
<div style="background:#f5f5f5; border:1px solid #dddddd; border-radius:12px; overflow:hidden; font-family:Georgia, serif;">
    <div style="background:#2f3b52; color:white; padding:18px 24px;">
        <h2 style="margin:0; font-size:24px;">Produce Category Strategy Card</h2>
        <p style="margin:6px 0 0; opacity:0.85; font-size:14px;">Organic vs. Conventional Segment Review & Berry Campaign Analysis</p>
    </div>
    <div style="padding:20px 24px 8px;">
        <h3 style="margin:0 0 12px; color:#2f3b52;">Segment Overview: Organic vs. Conventional</h3>
        <table style="width:100%; border-collapse:collapse; font-size:14px; margin-bottom:20px;">
            <thead>
                <tr style="background:#2f3b52; color:white;">
                    <th style="padding:8px 12px; text-align:left;">Segment</th>
                    <th style="padding:8px 12px; text-align:right;">Product Count</th>
                    <th style="padding:8px 12px; text-align:right;">Total Orders</th>
                    <th style="padding:8px 12px; text-align:right;">Reorder Rate</th>
                </tr>
            </thead>
            <tbody>
                <tr style="background:#eeeeee;">
                    <td style="padding:8px 12px;"><b>Organic Produce</b></td>
                    <td style="padding:8px 12px; text-align:right;">{org_prods:,}</td>
                    <td style="padding:8px 12px; text-align:right;">{org_orders:,}</td>
                    <td style="padding:8px 12px; text-align:right;"><b>{org_reorder:.2%}</b></td>
                </tr>
                <tr>
                    <td style="padding:8px 12px;"><b>Conventional Produce</b></td>
                    <td style="padding:8px 12px; text-align:right;">{conv_prods:,}</td>
                    <td style="padding:8px 12px; text-align:right;">{conv_orders:,}</td>
                    <td style="padding:8px 12px; text-align:right;">{conv_reorder:.2%}</td>
                </tr>
            </tbody>
        </table>
        <h3 style="margin:16px 0 12px; color:#2f3b52;">Category Benchmark Staples</h3>
        <div style="display:flex; gap:16px; margin-bottom:20px;">
            <div style="flex:1; background:#ffffff; padding:14px 16px; border-radius:8px; border:1px solid #e0e0e0; border-top:4px solid #2f3b52;">
                <div style="font-size:11px; text-transform:uppercase; color:#777777; font-weight:bold; margin-bottom:4px;">Conventional Staple</div>
                <div style="font-size:18px; font-weight:bold; color:#000000;">Banana</div>
                <div style="font-size:12px; color:#888888;">Product ID: 24852</div>
            </div>
            <div style="flex:1; background:#ffffff; padding:14px 16px; border-radius:8px; border:1px solid #e0e0e0; border-top:4px solid #2e7d32;">
                <div style="font-size:11px; text-transform:uppercase; color:#2e7d32; font-weight:bold; margin-bottom:4px;">Organic Staple</div>
                <div style="font-size:18px; font-weight:bold; color:#000000;">{organic_title}</div>
                <div style="font-size:12px; color:#888888;">Product ID: {organic_id}</div>
            </div>
        </div>
        <div style="background:#ffffff; padding:16px; border-radius:8px; border:1px solid #dddddd; margin-bottom:20px;">
            <h3 style="margin:0 0 10px; color:#2f3b52; font-size:16px;">Berry Promotion Campaign Review</h3>
            <table style="width:100%; border-collapse:collapse; font-size:14px; margin-bottom:10px;">
                <thead>
                    <tr style="background:#2f3b52; color:white;">
                        <th style="padding:8px 12px; text-align:left;">Berry Item</th>
                        <th style="padding:8px 12px; text-align:center;">Product ID</th>
                        <th style="padding:8px 12px; text-align:right;">Total Orders</th>
                        <th style="padding:8px 12px; text-align:right;">Reorder Rate</th>
                    </tr>
                </thead>
                <tbody>
                    <tr style="background:#eeeeee;">
                        <td style="padding:8px 12px;"><b>Organic Strawberries</b></td>
                        <td style="padding:8px 12px; text-align:center; color:#666666;">21137</td>
                        <td style="padding:8px 12px; text-align:right;">{org_straw_orders:,}</td>
                        <td style="padding:8px 12px; text-align:right;"><b>{org_straw_reorder:.2%}</b></td>
                    </tr>
                    <tr>
                        <td style="padding:8px 12px;"><b>Strawberries (Ordinary)</b></td>
                        <td style="padding:8px 12px; text-align:center; color:#666666;">16797</td>
                        <td style="padding:8px 12px; text-align:right;">{conv_straw_orders:,}</td>
                        <td style="padding:8px 12px; text-align:right;">{conv_straw_reorder:.2%}</td>
                    </tr>
                </tbody>
            </table>
            <p style="margin:0; font-size:13px; color:#555555; line-height:1.5;">
                <i>Analysis Note: Organic strawberries generate nearly 1.85x higher order volume and stronger repeat loyalty ({org_straw_reorder:.1%} vs. {conv_straw_reorder:.1%}) compared to ordinary strawberries, supporting premium promotion placement.</i>
            </p>
        </div>
    </div>
    <div style="margin:0 24px 22px; padding:16px 18px; background:#ffffff; border-left:5px solid #2f3b52; border-radius:8px;">
        <h3 style="margin:0 0 8px; color:#2f3b52;">Strategic Recommendation</h3>
        <p style="margin:0; line-height:1.65; font-size:15px;">{nb2_outlook}</p>
    </div>
    <div style="background:#eeeeee; color:#777777; padding:10px 24px; font-size:12px; font-style:italic; text-align:right;">Source: Instacart product catalog</div>
</div>
"""
display(HTML(card_html))


Segment,Product Count,Total Orders,Reorder Rate
Organic Produce,473,"5,113,064",67.10%
Conventional Produce,"1,211","4,366,227",62.52%
Berry Item,Product ID,Total Orders,Reorder Rate
Organic Strawberries,21137,"264,683",77.77%
Strawberries (Ordinary),16797,"142,951",69.82%


## Conclusion

The analysis verifies that Organic produce outperforms Conventional produce in total orders (5,113,064 vs. 4,366,227) and reorder rate (67.10% vs. 62.52%). Selecting Organic Banana (Product ID: 37067) as the organic produce staple allows rendering the requested produce strategy card.